In [ ]:
# !pip install selenium
# !pip install webdriver-manager
# !pip install python-dotenv
# !pip install pandas

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.5 MB 4.2 MB/s eta 0:00:03
   ------ --------------------------------- 1.6/9.5 MB 4.0 MB/s eta 0:00:03
   ---------- ----------------------------- 2.6/9.5 MB 4.2 MB/s eta 0:00:02
   ---------------- ----------------------- 3.9/9.5 MB 5.0 MB/s eta 0:00:02
   ------------------------ --------------- 5.8/9.5 MB 5.7 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.5 MB 6.0 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.5 MB 6.5 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 6.5 MB/s  0:00:01

   ----------------------------------------  0/17 [sortedcontainers]
   -- -------------------------------------  1/17 [websocket-client]
   -- -------------------------------------  1/17 [websocket-client]
   -- -------------------------------------  1/17 [websocket-client]
   -- -------------------------------------  1/17


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0/4 [python-dotenv]
   ---------------------------------------- 0/4 [python-dotenv]
   ---------------------------------------- 0/4 [python-dotenv]
   ---------------------------------------- 0/4 [python-dotenv]
   ---------- ----------------------------- 1/4 [charset_normalizer]
   ---------- ----------------------------- 1/4 [charset_normalizer]
   ---------- ----------------------------- 1/4 [charset_normalizer]
   ---------- ----------------------------- 1/4 [charset_normalizer]
   -------------------- ------------------- 2/4 [requests]
   -------------------- ------------------- 2/4 [requests]
   -------------------- ------------------- 2/4 [requests]
   -------------------- ------------------- 2/4 [requests]
   ------------------------------ --------- 3/4 [webdriver-manager]
   ------------------------------ --------- 3/4 [webdriver-manager]
   ------------------------------ --------- 3/4 [webdriver-manager]
   ------------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.9 MB 4.5 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/9.9 MB 5.0 MB/s eta 0:00:02
   --------------- ------------------------ 3.9/9.9 MB 5.4 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/9.9 MB 5.8 MB/s eta 0:00:01
   --------------------------- ------------ 6.8/9.9 MB 6.1 MB/s eta 0:00:01
   ---------------------------------- ----- 8.4/9.9 MB 6.1 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 6.6 MB/s  0:00:01
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/12.4 MB 10.6 MB/s eta 0:00:02
   ---------- ----------------------------- 3.1/12.4 MB 9.4 MB/s eta 0:00:01
   ------------- -------------------------- 4.2/12.4 MB 7.4 MB/s eta 0:00:02
   ----------------- ----------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from datetime import datetime
import zipfile
from dotenv import load_dotenv
import pandas as pd

In [14]:
env_path = os.path.join(os.path.dirname(os.getcwd()), '.env')
load_dotenv(dotenv_path=env_path, override=True)

username = os.getenv("MY_USERNAME")
password = os.getenv("MY_PASSWORD")

url_download = "https:/nbr.chnp.co.th"

# 1. หา Parent Directory (ถอยกลับไป 1 ชั้นจากที่รัน script)
parent_dir = os.path.dirname(os.getcwd())


# 2. ตั้งชื่อโฟลเดอร์ตามวันที่ปัจจุบัน
now = datetime.now() # สร้างตัวแปร now ไว้ใช้ร่วมกัน
today = now.strftime('%Y-%m-%d')

# 3. รวม Path...
download_path = os.path.join(parent_dir, "tmp", today)
os.makedirs(download_path, exist_ok=True)

# --- ส่วนที่แก้ไข ---
# ใช้ f-string ดึงปี (Y) และเดือน (m) มาต่อกันเป็น 202602
current_month_suffix = now.strftime('%Y%m') 
target_file = f"Port_Complete_{current_month_suffix}.zip" 

# หมายเหตุ: หากไฟล์มีนามสกุล อย่าลืมเติมด้วยนะครับ เช่น 
# target_file = f"Port_Complete_{current_month_suffix}.zip"
# ------------------

print(f"📁 ไฟล์จะถูกดาวน์โหลดไปที่: {download_path}")
print(f"🎯 ชื่อไฟล์เป้าหมายคือ: {target_file}")

chrome_options = Options()

# 2. add Prefs
prefs = {
    "download.default_directory": download_path, # ต้องเป็น Absolute Path
    "download.prompt_for_download": False,        # ห้ามถามที่เซฟ
    "download.directory_upgrade": True,
    "safebrowsing.enabled": False,                # ปิด Safe Browsing ชั่วคราว (เพื่อไม่ให้มันกักไฟล์)
    "plugins.always_open_pdf_externally": True,  # ปิดการเปิดไฟล์ในเบราว์เซอร์
    "profile.default_content_settings.popups": 0, # ปิด Popup ทุกชนิด
}
chrome_options.add_experimental_option("prefs", prefs)

# 3. เพิ่ม Arguments เพื่อปิดระบบความปลอดภัยที่ขวางการโหลด
chrome_options.add_argument("--disable-features=InsecureDownloadWarnings")
chrome_options.add_argument("--ignore-certificate-errors")


📁 ไฟล์จะถูกดาวน์โหลดไปที่: c:\Users\Kanok Onteaun\Desktop\TRUE\Project\MNP_Dashboard\tmp\2026-02-25
🎯 ชื่อไฟล์เป้าหมายคือ: Port_Complete_202602.zip


In [11]:
# --- Start Browser ---
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

try:
    driver.get(url_download)
    wait = WebDriverWait(driver, 20) # เพิ่มเวลาเผื่อเน็ตช้า

 # 1. รอให้ช่อง Username ปรากฏ (ใช้ความยืดหยุ่นของ XPath)
    user_xpath = "//input[@type='text' or contains(@placeholder, 'User')]"
    user_input = wait.until(EC.element_to_be_clickable((By.XPATH, user_xpath)))
    
    # 2. กรอกข้อมูล (ล้างค่าเก่าก่อน)
    user_input.clear()
    user_input.send_keys(username)
    
    pass_input = driver.find_element(By.XPATH, "//input[@type='password']")
    pass_input.clear()
    pass_input.send_keys(password)

    # 3. คลิกปุ่ม Log In (ปรับให้ตรงตามรูปที่มีช่องว่าง 'Log In')
    # เราจะหา Element ที่มีข้อความ "Log In" ไม่ว่าจะเป็น tag อะไรก็ตาม
    login_btn_xpath = "//*[self::button or self::a or self::div][contains(text(), 'Log In')]"
    login_btn = wait.until(EC.element_to_be_clickable((By.XPATH, login_btn_xpath)))

    # เลื่อนหน้าจอไปที่ปุ่ม (เผื่อปุ่มอยู่ขอบจอ)
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", login_btn)
    time.sleep(0.5)

    try:
        # พยายามคลิกแบบปกติก่อน
        login_btn.click()
    except:
        # ถ้าคลิกปกติไม่ได้ ให้ใช้ JavaScript Force Click
        driver.execute_script("arguments[0].click();", login_btn)

    print("✅ กดปุ่ม Log In สำเร็จ")

    side_menu_xpath = "//a[contains(@href, 'page=downloaddetail')]"
    
    menu_btn = wait.until(EC.element_to_be_clickable((By.XPATH, side_menu_xpath)))
    
    # 2. เลื่อนหน้าจอไปที่เมนู
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", menu_btn)
    time.sleep(0.5)
    
    # 3. คลิกด้วย JavaScript (ช่วยแก้ปัญหาเวลาเมนูโดน Element อื่นซ้อนทับ)
    driver.execute_script("arguments[0].click();", menu_btn)
    
    print("✅ เข้าสู่เมนู Download Details สำเร็จ")
    
    # --- ส่วนที่เพิ่มเข้ามา: ลบไฟล์เดิมถ้ามีอยู่ ---
    full_path = os.path.join(download_path, target_file)
    if os.path.exists(full_path):
        os.remove(full_path)
        print(f"🗑️ ลบไฟล์เก่า {target_file} ออกเรียบร้อย เพื่อป้องกันชื่อซ้ำ")
    
    print("✅ เข้าสู่เมนู Download Details สำเร็จ")
    # 3. คลิกปุ่ม 'Download' สีเขียว (ตามรูปที่ส่งมา)
    download_btn_xpath = "//button[contains(., 'Download')]"
    download_btn = wait.until(EC.element_to_be_clickable((By.XPATH, download_btn_xpath)))
    
    # เลื่อนจอและคลิก
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", download_btn)
    time.sleep(1) 
    download_btn.click()
    
    print(f"🚀 เริ่มดาวน์โหลดไฟล์: {target_file}")

    # 4. ฟังก์ชันรอจนกว่าจะโหลดเสร็จ (คงเดิม)
    def wait_for_download(directory, timeout=60):
        seconds = 0
        while seconds < timeout:
            files = os.listdir(directory)
            # เช็คไฟล์ชั่วคราว
            if any(f.endswith('.crdownload') or f.endswith('.tmp') for f in files):
                time.sleep(1)
                seconds += 1
            else:
                # เช็คว่าไฟล์ที่ต้องการโผล่มาหรือยัง
                if target_file in files:
                    return True
                time.sleep(1)
                seconds += 1
        return False

    if wait_for_download(download_path):
        print(f"✅ ดาวน์โหลดสำเร็จ! ไฟล์อยู่ที่: {os.path.join(download_path, target_file)}")
    else:
        print("❌ การดาวน์โหลดล้มเหลว หรือไฟล์ชื่อไม่ตรงกับที่คาดไว้")

except Exception as e:
    print(f"เกิดข้อผิดพลาด: {e}")

finally:
    time.sleep(5) # ให้เวลา browser เคลียร์แคชแป๊บนึง
    driver.quit()

✅ กดปุ่ม Log In สำเร็จ
✅ เข้าสู่เมนู Download Details สำเร็จ
🗑️ ลบไฟล์เก่า Port_Complete_202602.zip ออกเรียบร้อย เพื่อป้องกันชื่อซ้ำ
✅ เข้าสู่เมนู Download Details สำเร็จ
🚀 เริ่มดาวน์โหลดไฟล์: Port_Complete_202602.zip
✅ ดาวน์โหลดสำเร็จ! ไฟล์อยู่ที่: c:\Users\Kanok Onteaun\Desktop\TRUE\Project\MNP_Dashboard\tmp\2026-02-25\Port_Complete_202602.zip


In [18]:
zip_file_path = os.path.join(download_path, target_file) 
file_password = os.getenv("ZIP_PASSWORD")

try:
    if os.path.exists(zip_file_path):
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            # ตรวจสอบว่ามีรหัสผ่านหรือไม่ ถ้ามีให้แปลงเป็น bytes
            pwd = file_password.encode('utf-8') if file_password else None
            
            print(f"📦 กำลังแตกไฟล์: {zip_file_path}")
            # ใส่รหัสผ่านในช่อง pwd
            zip_ref.extractall(path=download_path, pwd=pwd)
            
        print(f"✅ แตกไฟล์ลงใน {download_path} เรียบร้อยแล้ว")
        
        # ลบไฟล์ ZIP ต้นฉบับทิ้งหลังจากแตกไฟล์สำเร็จเท่านั้น
        os.remove(zip_file_path)
        print(f"🗑️ ลบไฟล์ ZIP ต้นฉบับเรียบร้อยแล้ว")
    else:
        print("❌ ไม่พบไฟล์ ZIP ที่ต้องการแตก")

except RuntimeError as e:
    if "password" in str(e).lower():
        print("❌ รหัสผ่านไม่ถูกต้อง หรือไฟล์นี้ต้องใช้รหัสผ่าน")
    else:
        print(f"❌ เกิดข้อผิดพลาดขณะแตกไฟล์: {e}")
except zipfile.BadZipFile:
    print("❌ ไฟล์ ZIP เสียหรือโหลดมาไม่สมบูรณ์")
except Exception as e:
    print(f"❌ เกิดข้อผิดพลาดที่ไม่คาดคิด: {e}")




📦 กำลังแตกไฟล์: c:\Users\Kanok Onteaun\Desktop\TRUE\Project\MNP_Dashboard\tmp\2026-02-25\Port_Complete_202602.zip
✅ แตกไฟล์ลงใน c:\Users\Kanok Onteaun\Desktop\TRUE\Project\MNP_Dashboard\tmp\2026-02-25 เรียบร้อยแล้ว
🗑️ ลบไฟล์ ZIP ต้นฉบับเรียบร้อยแล้ว


In [ ]:
import pandas as pd
csv_filename = os.path.join(download_path, target_file.replace(".zip",".csv")) 

df = pd.read_csv(csv_filename)

In [31]:
df = df.astype({
    'MSISDN': 'string',
    'OrderID': 'string',
    'Recipient': 'string',
    'Rmcode': 'string',
    'Donor': 'string',
    'Dmcode' : 'string'
})

df['Rmcode'] = df['Rmcode'].astype(str).str.zfill(2)
df['Dmcode'] = df['Dmcode'].astype(str).str.zfill(2)

df['PortDate'] = pd.to_datetime(df['PortDate'])

df.info()
df.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41774 entries, 0 to 41773
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   PortDate   41774 non-null  datetime64[ns]
 1   MSISDN     41774 non-null  string        
 2   OrderID    41774 non-null  string        
 3   Recipient  41774 non-null  string        
 4   Rmcode     41774 non-null  object        
 5   Donor      41774 non-null  string        
 6   Dmcode     41774 non-null  object        
dtypes: datetime64[ns](1), object(2), string(4)
memory usage: 2.2+ MB


,PortDate,MSISDN,OrderID,Recipient,Rmcode,Donor,Dmcode
0,2026-02-02 04:00:00,610085805,32601300668372,AWN,03,TUC,06
1,2026-02-02 04:00:00,610133127,32601300668609,AWN,03,TUC,06
2,2026-02-02 04:00:00,610150208,32601290668184,AWN,03,TUC,06
3,2026-02-02 04:00:00,610285782,62601301802000,TUC,06,AWN,03
4,2026-02-02 04:00:00,610358482,62601301802300,TUC,06,AWN,03


In [88]:
schema = {
    "First Name": "string",
    "Last Name" : "string",
    "Company": "string",
    "City": "string",
    "Country": "string",
    "Phone 1": "string",
    "Phone 2": "string",
    "Email": "string",
    "Subscription Date": "datetime64[ns]",
    "Website": "string"
}

def cast_and_align(df, schema):
    df = df.copy()
    
    for col, dtype in schema.items():
        if "datetime" in dtype:
            df[col] = pd.to_datetime(df[col], errors="coerce")
        elif dtype in ["float64", "int64"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        else:
            df[col] = df[col].astype(dtype)
    
    return df[list(schema.keys())]

df_clean = cast_and_align(df, schema)